# 03 - Seeing How the Agent Thinks

`result.final_output` only shows you the *final* answer. But a lot happens before that: the agent might call a tool, look at the result, and decide to call another tool, before finally replying. This notebook is about seeing that process -- which is exactly what your Telegram bot will show its users later.


In [ ]:
%pip install -q openai-agents

from agents import set_default_openai_key

# Don't share or commit this notebook with your key filled in.
OPENAI_API_KEY = "sk-..."  # <-- paste your key here
set_default_openai_key(OPENAI_API_KEY)

print("Ready to go.")

## Looking inside a run

Every `Runner.run_sync(...)` result has a `.new_items` list -- everything that happened during that run, in order: tool calls, tool outputs, and the final message.


In [ ]:
from agents import Agent, Runner, function_tool

@function_tool
def get_current_time() -> str:
    """Return the current date and time."""
    from datetime import datetime

    return datetime.now().strftime("%A, %d %B %Y, %H:%M")


agent = Agent(
    name="Time-aware Assistant",
    instructions="Use your tools when they help answer the question.",
    tools=[get_current_time],
)

result = Runner.run_sync(agent, "What day of the week is it?")

for item in result.new_items:
    print(item.type)

You should see something like `tool_call_item`, then `tool_call_output_item`, then `message_output_item`. That's the agent calling `get_current_time`, seeing what it returned, and then writing its final reply.

## A simple trace printer

Let's write a small helper that prints this more readably -- this is a simplified version of what `app.py` in your capstone project does before sending you the bot's actual reply.


In [ ]:
def print_trace(result):
    for item in result.new_items:
        if item.type == "tool_call_item":
            name = item.tool_name or "tool"
            args = getattr(item.raw_item, "arguments", "")
            print(f"🔧 {name}({args})")
        elif item.type == "tool_call_output_item":
            print(f"   -> {item.output}")

    print(f"\n💬 {result.final_output}")


result = Runner.run_sync(agent, "What time is it?")
print_trace(result)

## Chaining multiple tools

Give the agent two tools and ask a question that genuinely needs both. Watch the trace -- it should call one, then the other, *before* replying.


In [ ]:
@function_tool
def convert_temperature(celsius: float) -> str:
    """Convert a Celsius temperature to Fahrenheit.

    Args:
        celsius: The temperature in Celsius.
    """
    fahrenheit = celsius * 9 / 5 + 32
    return f"{celsius}°C is {fahrenheit}°F"


agent = Agent(
    name="Multi-tool Assistant",
    instructions="Use your tools when they help answer the question.",
    tools=[get_current_time, convert_temperature],
)

result = Runner.run_sync(
    agent,
    "What's today's date, and also, what's 100 degrees Celsius in Fahrenheit?",
)
print_trace(result)

## This iteration is automatic

You didn't write a loop to make this happen -- it's built into the SDK. After every tool call, the model is automatically asked "given this result, what next?" and it can call another tool, or decide it has enough to answer. This continues for up to `max_turns` steps (10 by default, plenty for a course project) before it's forced to stop.

### Exercise

Add a third tool of your own, and craft a single question that requires the agent to use all three tools before answering. Confirm it with `print_trace`.


In [ ]:
# TODO: define a third @function_tool, add it to a new agent's
# tools list, and ask a question that needs all three tools.


**Next:** open `04_hosted_tools.ipynb`.
